In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import kagglehub
import numpy as np

print("Downloading dataset...")
path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")
base_dir = os.path.join(path, "PetImages")


Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.


In [ ]:

class TinyCDDataset(Dataset):
    def __init__(self, root_dir, limit=1000):
        self.files = []
        self.transform = transforms.Compose([
            transforms.Resize((8, 8)),
            transforms.Grayscale(),
            transforms.ToTensor(), # 0.0-1.0 float
        ])

        for label, sub in enumerate(["Cat", "Dog"]): # 0: Cat, 1: Dog
            dir_path = os.path.join(root_dir, sub)
            count = 0
            for f in os.listdir(dir_path):
                if f.endswith('.jpg') and count < limit:
                    try: # Проверка на битые файлы
                        full_path = os.path.join(dir_path, f)
                        Image.open(full_path).verify()
                        self.files.append((full_path, float(label)))
                        count += 1
                    except: pass

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        img = Image.open(path)
        return self.transform(img).view(-1), torch.tensor([label]) # Flatten 8x8 -> 64

dataset = TinyCDDataset(base_dir, limit=5000)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_data, test_data = torch.utils.data.random_split(dataset, [train_size, test_size])
loader = DataLoader(train_data, batch_size=16, shuffle=True)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [ ]:

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Вход 64 (8x8), Скрытый 8, Выход 1
        self.fc1 = nn.Linear(64, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.sigmoid(x)

model = SimpleNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [ ]:

print(f"Training on {len(train_data)} images...")
for epoch in range(10):
    total_loss = 0
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Training on 8000 images...
Epoch 1, Loss: 0.6900
Epoch 2, Loss: 0.6845
Epoch 3, Loss: 0.6808
Epoch 4, Loss: 0.6803
Epoch 5, Loss: 0.6814
Epoch 6, Loss: 0.6793
Epoch 7, Loss: 0.6792
Epoch 8, Loss: 0.6774
Epoch 9, Loss: 0.6777
Epoch 10, Loss: 0.6776


In [ ]:

print("\nTesting...")
model.eval()
correct = 0
with torch.no_grad():
    for inputs, labels in DataLoader(test_data, batch_size=1):
        output = model(inputs)
        pred = 1 if output.item() > 0.5 else 0
        if pred == labels.item():
            correct += 1
print(f"Accuracy: {correct/len(test_data)*100:.1f}%")
def quantize_layer(name, tensor):

    max_val = torch.max(torch.abs(tensor))
    scale = 127 / max_val.item()
    q_weights = (tensor * scale).round().int().numpy()

    print(f"\n[{name}] Scale: {scale:.4f}")
    print(f"Shape: {q_weights.shape}")
    
    print(f"Values: {list(q_weights.flatten())}")


for name, param in model.named_parameters():
    quantize_layer(name, param.data)


Testing...
Accuracy: 57.1%

[fc1.weight] Scale: 67.9364
Shape: (8, 64)
Values: [np.int32(-45), np.int32(7), np.int32(1), np.int32(-14), np.int32(-3), np.int32(8), np.int32(22), np.int32(-53), np.int32(-29), np.int32(28), np.int32(-25), np.int32(-127), np.int32(-126), np.int32(-7), np.int32(66), np.int32(-27), np.int32(-22), np.int32(58), np.int32(-25), np.int32(-66), np.int32(-82), np.int32(-11), np.int32(29), np.int32(-46), np.int32(-57), np.int32(35), np.int32(26), np.int32(47), np.int32(44), np.int32(50), np.int32(17), np.int32(-45), np.int32(-58), np.int32(-9), np.int32(15), np.int32(51), np.int32(52), np.int32(45), np.int32(-5), np.int32(-72), np.int32(-35), np.int32(-7), np.int32(38), np.int32(75), np.int32(86), np.int32(28), np.int32(-28), np.int32(-61), np.int32(-28), np.int32(-21), np.int32(21), np.int32(63), np.int32(61), np.int32(44), np.int32(0), np.int32(-34), np.int32(-38), np.int32(-16), np.int32(9), np.int32(63), np.int32(69), np.int32(31), np.int32(-5), np.int32(-7), 

In [ ]:

print("\n=== Exporting data for Vivado ===")


FRAC_BITS = 7
HEX_WIDTH = 16  # 16-битные числа

def float_to_hex(val):
    """Преобразует float в hex-строку в формате fixed-point int16"""
    
    scaled = int(round(val * (1 << FRAC_BITS)))

    scaled = max(min(scaled, 32767), -32768)

    hex_val = f"{(scaled & 0xFFFF):04x}"
    return hex_val

def export_tensor(tensor, filename):
    """Сохраняет тензор PyTorch в текстовый файл .hex"""
    data = tensor.detach().cpu().numpy().flatten()
    with open(filename, 'w') as f:
        for val in data:
            f.write(float_to_hex(val) + '\n')
    print(f"Saved {filename} (size: {len(data)})")


# Слой 1
export_tensor(model.fc1.weight, "w1.hex")
export_tensor(model.fc1.bias,   "b1.hex")
# Слой 2
export_tensor(model.fc2.weight, "w2.hex")
export_tensor(model.fc2.bias,   "b2.hex")


N_SAMPLES = 5
print(f"\nExporting {N_SAMPLES} test images...")

test_loader = DataLoader(test_data, batch_size=1, shuffle=False)
labels_hex = []

for i, (img, label) in enumerate(test_loader):
    if i >= N_SAMPLES:
        break

    fname = f"img{i}.hex"
    export_tensor(img[0], fname)

    lbl = int(label.item())
    labels_hex.append(f"{lbl:x}")

# Записываем файл с правильными ответами
with open("labels.hex", "w") as f:
    for l in labels_hex:
        f.write(l + "\n")
print("Saved labels.hex")

print("\n=== Export Done! ===")
print("Теперь скопируйте файлы *.hex в папку симуляции Vivado.")


=== Exporting data for Vivado ===
Saved w1.hex (size: 512)
Saved b1.hex (size: 8)
Saved w2.hex (size: 8)
Saved b2.hex (size: 1)

Exporting 5 test images...
Saved img0.hex (size: 64)
Saved img1.hex (size: 64)
Saved img2.hex (size: 64)
Saved img3.hex (size: 64)
Saved img4.hex (size: 64)
Saved labels.hex

=== Export Done! ===
Теперь скопируйте файлы *.hex в папку симуляции Vivado.
